### Native Bayes

In [14]:
import pandas as pd
import time
import warnings
import tracemalloc
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score
from scipy.stats import loguniform

In [15]:
def native_bayes(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    nb = GaussianNB()

    tracemalloc.start()
    start_time = time.time()

    nb.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    y_pred = nb.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb

#### Mетрики без подбора гиперпараметров

In [16]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage = native_bayes(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.938
Среднее время = 0.00172 сек
Среднее потребление памяти = 0.207 MB


,samples (n),features (m),f1-score,time (sec),memory (MB)
0,100,5,1.000000,0.002584,0.017052
1,100,8,0.967742,0.002928,0.020363
2,100,11,0.857143,0.002067,0.038536
3,500,5,0.974026,0.001634,0.066666
4,500,8,0.935484,0.001535,0.099281
5,500,11,0.884848,0.001519,0.162910
6,1000,5,0.992754,0.001681,0.129379
7,1000,8,0.967509,0.001344,0.193459
8,1000,11,0.868421,0.001311,0.279541
9,3000,5,0.984281,0.001264,0.308161


Наивный Байесовский классификатор показывает `f1-score` больший, чем в Linear SVC, но требует немного больше памяти и меньше времени.

#### Mетрики с подбором гиперпараметров

In [17]:
def native_bayes_params(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    param_dist = {
        'var_smoothing': loguniform(1e-10, 1e-7) 
    }       # сглаживание

    nb = GaussianNB()

    random_search = RandomizedSearchCV(
        nb, param_dist, n_iter=10, cv=5, scoring='f1', random_state=81
    )

    tracemalloc.start()
    start_time = time.time()

    random_search.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    best_params = random_search.best_params_
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb, best_params, best_model

In [18]:
results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage, best_p, model = native_bayes_params(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage,
            'var_smoothing': best_p['var_smoothing']
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.938
Среднее время = 0.23098 сек
Среднее потребление памяти = 0.384 MB


,samples (n),features (m),f1-score,time (sec),memory (MB),var_smoothing
0,100,5,1.000000,0.221229,0.076861,1.164816e-09
1,100,8,0.967742,0.218389,0.082867,1.164816e-09
2,100,11,0.857143,0.215500,0.103735,1.164816e-09
3,500,5,0.974026,0.213154,0.160383,1.164816e-09
4,500,8,0.935484,0.217688,0.196185,1.164816e-09
5,500,11,0.884848,0.221140,0.275649,1.164816e-09
6,1000,5,0.992754,0.225588,0.271029,1.164816e-09
7,1000,8,0.967509,0.287298,0.341471,1.164816e-09
8,1000,11,0.868421,0.261119,0.474420,1.164816e-09
9,3000,5,0.984281,0.228579,0.626115,1.164816e-09


После подбора гиперпараметра `f1-score` не изменился. 

Параметр `var_smoothing` отвечает за вычислительную стабильность модели. Он добавляет небольшое значение к дисперсии признаков, чтобы расширить границы нормального распределения. Это предотвращает ситуацию, когда модель приписывает нулевую вероятность редким событиям.

### Сохранение модели

In [20]:
import joblib

joblib.dump(model, 'models/native_bayes_model.pkl')

['models/native_bayes_model.pkl']